In [11]:
"""
Character-Level Transformer for Sequence Memorization
----------------------------------------------------
This program implements a Mini-Transformer model designed to learn and
reconstruct multiple specific sentences at a character level. It uses
causal self-attention to predict the next character in a sequence,
allowing it to "complete" a sentence given a starting prompt.

"""

import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np


class CharTokenizer:

    """Handles converting text into integer tokens and vice-versa at the character level"""

    def __init__(self, text):

        """Initializes vocabulary based on unique characters in the input text"""

        self.chars = sorted(list(set(text)))
        self.char2idx = {c: i for i, c in enumerate(self.chars)}
        self.idx2char = {i: c for i, c in enumerate(self.char2idx)}
        self.vocab_size = len(self.chars)

    def encode(self, text):

       """Converts a string into a list of character indices"""

       return [self.char2idx[c] for c in text]

    def decode(self, tokens):

       """Converts a list of character indices back into a string"""

       return "".join([self.idx2char[t] for t in tokens])


class TransformerBlock(layers.Layer):

    """A single Transformer layer using Multi-Head Attention and Feed-Forward networks."""

    def __init__(self, embed_dim, num_heads):

        """Sets up attention, feed-forward, and normalization layers."""

        super().__init__()
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            layers.Dense(embed_dim * 4, activation="relu"),
            layers.Dense(embed_dim)
        ])
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()

    def call(self, x, training=False):

        """Applies causal self-attention followed by a feed-forward network with residual connections."""

        attn_output = self.mha(query=x, value=x, key=x, use_causal_mask=True, training=training)
        x = self.layernorm1(x + attn_output)
        return self.layernorm2(x + self.ffn(x))


class MultiSentenceTransformer(Model):

    """A Mini-Transformer model designed to learn and predict multiple short sentences."""

    def __init__(self, vocab_size, seq_len):

        """Initializes embeddings, transformer blocks, and the final output head."""

        super().__init__()
        self.embed_dim = 128
        self.embedding = layers.Embedding(vocab_size, self.embed_dim)
        self.pos_emb = layers.Embedding(seq_len, self.embed_dim)
        self.blocks = [TransformerBlock(self.embed_dim, 4) for _ in range(2)]
        self.head = layers.Dense(vocab_size)

    def call(self, x, training=False):

        """Processes a sequence of tokens to produce logits for the next character in the sequence."""

        T = tf.shape(x)[1]
        positions = tf.range(start=0, limit=T, delta=1)
        x = self.embedding(x) + self.pos_emb(positions)
        for block in self.blocks:
            x = block(x, training=training)
        return self.head(x)

# DATA SET
sentences = [
    "the quick brown fox jumps over the lazy dog.",
    "the cat sat on the mat and slept all day.",
    "coding in python is fun and very powerful.",
    "tensorflow makes building models easy."
]
all_text = " ".join(sentences)
tokenizer = CharTokenizer(all_text)

SEQ_LEN = 15
X, Y = [], []

for s in sentences:
    encoded = tokenizer.encode(s)
    for i in range(len(encoded) - SEQ_LEN):
        X.append(encoded[i : i + SEQ_LEN])
        Y.append(encoded[i + 1 : i + SEQ_LEN + 1])

X, Y = np.array(X), np.array(Y)

model = MultiSentenceTransformer(tokenizer.vocab_size, SEQ_LEN)
model.compile(optimizer="adam", loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))

# TRAINING
print("Training...")
model.fit(X, Y, epochs=120, verbose=0)
print("Training Complete.\n")

# PREDICTION
def predict_sentence(start_str, length=45):


    """Generates a text completion based on a provided starting string."""

    tokens = tokenizer.encode(start_str)
    for _ in range(length):
        inp = np.array([tokens[-SEQ_LEN:]])
        logits = model(inp, training=False)
        next_token = tf.argmax(logits[0, -1, :], axis=-1).numpy()
        tokens.append(int(next_token))
        if tokenizer.decode([next_token]) == ".": break
    return tokenizer.decode(tokens)

print("--- Testing Predictions ---")
for test in ["the quick","coding"]:
    print(f"Input: '{test}' -> {predict_sentence(test)}")


Training...
Training Complete.

--- Testing Predictions ---
Input: 'the quick' -> the quick brown fox jumps over the lazy dog.
Input: 'coding' -> coding in python is fun and very powerful.
